In [22]:
from dotenv import load_dotenv
import os
from pinecone import Pinecone, ServerlessSpec
import time
import pypdf
from transformers import AutoModel
from pypdf import PdfReader
load_dotenv()

True

In [23]:
pinecone_api_key = os.environ.get("PINECONE_API_KEY")
groq_api_key = os.environ.get("GROQ_API_KEY")

In [24]:
def chunk_text(text, size=100):
    return [text[i:i+size] for i in range(0, len(text), size)]

In [ ]:
model = AutoModel.from_pretrained('jinaai/jina-embeddings-v2-base-es', trust_remote_code=True) # trust_remote_code is needed to use the encode method

In [ ]:
pc = Pinecone(api_key=pinecone_api_key)

spec = ServerlessSpec(cloud="aws", region="us-east-1")

# choose a name for your index
index_name = "cv-search"

In [ ]:
# check if index already exists (it shouldn't if this is first time)
if index_name not in pc.list_indexes().names():
    # if does not exist, create index
    pc.create_index(
        index_name,
        dimension=768,
        metric='dotproduct',
        spec=spec
    )
    # wait for index to be initialized
    while not pc.describe_index(index_name).status['ready']:
        time.sleep(1)

# connect to index
index = pc.Index(index_name)
# view index stats
index.describe_index_stats()

In [34]:
import hashlib

In [35]:

file_names = ["Danilo_Reitano_CV_Development.pdf", "CV_RodrigoMesa.pdf", "Curriculum Vitae.pdf"]
person_name = {"Danilo_Reitano_CV_Development.pdf":"Danilo Reitano", "CV_RodrigoMesa.pdf":"Rodrigo Mesa", "Curriculum Vitae.pdf":"Juan Garcia"}
for file in file_names:
    texts = []
    reader = PdfReader(f"cvs/{file}")
    number_of_pages = len(reader.pages)
    for page_number in range(number_of_pages):
        page = reader.pages[page_number]
        text = page.extract_text()
        texts.extend(chunk_text(text))
    embeddings = model.encode(texts)
    metadata = {"person_name":person_name[file]}
    vectors = [{"id": hashlib.md5(f"{idx}{file}".encode()).hexdigest(),"values":data,"metadata":metadata} for idx,data in enumerate(embeddings)]
    index.upsert(
        vectors=vectors,
        namespace="vs_namespace"
    )